# LeetCode 373 — Find K Pairs with Smallest Sums bằng Python

Link đề: https://leetcode.com/problems/find-k-pairs-with-smallest-sums/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Heap / Priority Queue (hàng đợi ưu tiên), Binary Search (tìm kiếm nhị phân)

Skeleton LeetCode:

```python
class Solution:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ. Bài này có **nhiều đáp án đúng** (các cặp có cùng tổng có thể xếp theo thứ tự khác nhau), nên khi test ta dùng **hàm kiểm tra tính hợp lệ** thay vì so sánh từng phần tử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai mảng **đã sắp xếp tăng dần** `nums1`, `nums2` và số `k`; một **cặp** `(u, v)` lấy `u` từ `nums1` và `v` từ `nums2`; phải trả về **`k` cặp có tổng `u + v` nhỏ nhất**.
2. Nắm các công cụ Python: **`heapq`** (`heappush`, `heappop`, `heapify`, `heapreplace`, `nsmallest`), so sánh **tuple** (`(tổng, i, j)`), `set` đánh dấu đã thăm, `itertools.product`, `bisect`, `sorted(key=...)`.
3. Đi từ cách **sinh mọi cặp rồi sắp xếp** → **giữ `k` cặp nhỏ nhất bằng max-heap** → **min-heap "mở rộng biên" theo từng hàng** (cách nộp) → **min-heap kèm `visited` từ góc `(0, 0)`** → **tìm nhị phân trên giá trị tổng**.
4. Hiểu mấu chốt: ma trận ảo `S[i][j] = nums1[i] + nums2[j]` **tăng theo hàng và theo cột**, nên cặp nhỏ nhất kế tiếp luôn nằm ở **"biên"** của những cặp đã lấy — ta chỉ cần giữ biên trong heap thay vì mọi `m · n` cặp.
5. Tránh các lỗi kinh điển: **đẩy mọi cặp vào heap** (quá nhiều), **tràn chỉ số** khi mở rộng `j + 1` hoặc `i + 1`, **trùng cặp** khi không dùng `visited`, **bỏ sót cặp** khi chỉ mở rộng theo một hướng, và **so sánh sai** khi kiểm tra đáp án (nhiều đáp án đúng).

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2` đã **sắp xếp tăng dần (non-decreasing)** và một số nguyên `k`.

Một **cặp** `(u, v)` gồm **một phần tử của `nums1`** và **một phần tử của `nums2`**.

Hãy trả về **`k` cặp `(u1, v1), (u2, v2), ..., (uk, vk)` có tổng nhỏ nhất**.

**Ví dụ 1**

- Input: `nums1 = [1,7,11]`, `nums2 = [2,4,6]`, `k = 3`
- Output: `[[1,2],[1,4],[1,6]]`
- Giải thích: các cặp theo thứ tự tổng tăng dần: `[1,2],[1,4],[1,6],[7,2],[7,4],[11,2],[7,6],[11,4],[11,6]`; lấy `3` cặp đầu.

**Ví dụ 2**

- Input: `nums1 = [1,1,2]`, `nums2 = [1,2,3]`, `k = 2`
- Output: `[[1,1],[1,1]]`
- Giải thích: có hai phần tử `1` ở `nums1` ghép với `1` ở `nums2` ⇒ hai cặp `[1,1]` (tổng `2`).

**Ví dụ tự dựng (đối chiếu lại đề gốc)**

- `nums1 = [1,2]`, `nums2 = [3]`, `k = 2` → `[[1,3],[2,3]]`.
- `nums1 = [1,1,2]`, `nums2 = [1,2,3]`, `k = 9` → toàn bộ `9` cặp, theo tổng tăng dần.

**Ràng buộc**

- `1 <= nums1.length, nums2.length <= 10^5`
- `-10^9 <= nums1[i], nums2[i] <= 10^9`
- `nums1` và `nums2` đều **không giảm**.
- `1 <= k <= 10^4`
- `k <= nums1.length * nums2.length`

Chú ý: `m · n` có thể tới `10^10` cặp — **không thể** sinh hết. Nhưng `k` chỉ tối đa `10^4`: ta chỉ nên "chạm" tới khoảng `k` cặp.

In [1]:
from typing import List
import heapq
import itertools
import random
from bisect import bisect_left, bisect_right
from collections import Counter

examples = [
    ([1, 7, 11], [2, 4, 6], 3, [[1, 2], [1, 4], [1, 6]]),
    ([1, 1, 2], [1, 2, 3], 2, [[1, 1], [1, 1]]),
    ([1, 2], [3], 2, [[1, 3], [2, 3]]),
]
for nums1, nums2, k, expected in examples:
    print(f"nums1={nums1}, nums2={nums2}, k={k} -> kỳ vọng {expected}")

nums1=[1, 7, 11], nums2=[2, 4, 6], k=3 -> kỳ vọng [[1, 2], [1, 4], [1, 6]]
nums1=[1, 1, 2], nums2=[1, 2, 3], k=2 -> kỳ vọng [[1, 1], [1, 1]]
nums1=[1, 2], nums2=[3], k=2 -> kỳ vọng [[1, 3], [2, 3]]


## 3. Kiến thức nền

### 3.1. Mảng đã sắp xếp và ma trận tổng "ảo"

Vì `nums1` và `nums2` tăng dần, ta có thể hình dung một **ma trận** `S` với `S[i][j] = nums1[i] + nums2[j]` (không cần lưu thật). Tính chất quan trọng: **`S` tăng (không giảm) dọc theo mỗi hàng và dọc theo mỗi cột**, vì `nums2[j] <= nums2[j+1]` và `nums1[i] <= nums1[i+1]`.

In [2]:
nums1, nums2 = [1, 7, 11], [2, 4, 6]
S = [[u + v for v in nums2] for u in nums1]
for row in S:
    print(row)
print("tăng dọc hàng:", all(S[i][j] <= S[i][j + 1] for i in range(3) for j in range(2)))
print("tăng dọc cột :", all(S[i][j] <= S[i + 1][j] for i in range(2) for j in range(3)))

[3, 5, 7]
[9, 11, 13]
[13, 15, 17]
tăng dọc hàng: True
tăng dọc cột : True


### 3.2. `heapq`: hàng đợi ưu tiên (min-heap)

`heapq` của Python là **min-heap**: phần tử **nhỏ nhất luôn ở `heap[0]`**.

- `heapq.heappush(h, x)`: thêm `x`, `O(log n)`.
- `heapq.heappop(h)`: lấy ra và trả phần tử nhỏ nhất, `O(log n)`.
- `heapq.heapify(lst)`: biến list thành heap tại chỗ, `O(n)`.
- `heapq.heapreplace(h, x)`: pop rồi push `x` (nhanh hơn làm hai lần).
- `heapq.nsmallest(k, iterable, key=...)`: `k` phần tử nhỏ nhất, đã sắp xếp tăng dần.

Để có **max-heap**, lưu **số đối** (`-x`) vì heap luôn lấy phần tử nhỏ nhất.

In [3]:
h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
print("heap[0] =", h[0], "| list nội bộ:", h)
print("pop lần lượt:", [heapq.heappop(h) for _ in range(4)])

lst = [9, 4, 7, 1]
heapq.heapify(lst)
print("sau heapify:", lst, "| nhỏ nhất:", lst[0])
print("nsmallest:", heapq.nsmallest(2, [9, 4, 7, 1]))

mx = []
for x in [5, 1, 8, 3]:
    heapq.heappush(mx, -x)                 # max-heap bằng số đối
print("max-heap: lớn nhất =", -mx[0])

heap[0] = 1 | list nội bộ: [1, 3, 8, 5]
pop lần lượt: [1, 3, 5, 8]
sau heapify: [1, 4, 7, 9] | nhỏ nhất: 1
nsmallest: [1, 4]
max-heap: lớn nhất = 8


### 3.3. Tuple trong heap: so sánh từng phần tử từ trái sang phải

Heap có thể chứa **tuple** như `(tổng, i, j)`. Python so sánh tuple **từ trái sang phải**: so `tổng` trước; nếu bằng thì so `i`; nếu vẫn bằng thì so `j`. Nhờ vậy, ta vừa sắp xếp theo tổng, vừa mang theo **chỉ số** `(i, j)` để biết phải mở rộng ở đâu. Hệ quả: khi hai tổng bằng nhau, cặp có `i` nhỏ hơn được lấy trước — vẫn là đáp án hợp lệ.

In [4]:
print((3, 0, 1) < (3, 1, 0), (3, 2, 0) < (3, 1, 5), (2, 9, 9) < (3, 0, 0))
h = []
for t in [(7, 1, 0), (3, 2, 0), (3, 0, 1), (5, 0, 0)]:
    heapq.heappush(h, t)
print([heapq.heappop(h) for _ in range(4)])

True False True
[(3, 0, 1), (3, 2, 0), (5, 0, 0), (7, 1, 0)]


### 3.4. `itertools.product`, `set` và `sorted(key=...)`

- `itertools.product(a, b)` sinh **mọi cặp** `(x, y)` lười biếng (không tạo list lớn).
- `set` lưu các cặp chỉ số đã thăm: kiểm tra `(i, j) in visited` trung bình `O(1)`.
- `sorted(items, key=lambda p: p[0] + p[1])` sắp xếp theo **tổng**; `sort` của Python **ổn định**: phần tử bằng khóa giữ nguyên thứ tự ban đầu.

In [5]:
pairs = list(itertools.product([1, 7], [2, 4]))
print(pairs)
print(sorted(pairs, key=lambda p: p[0] + p[1]))

visited = {(0, 0)}
print((0, 0) in visited, (0, 1) in visited)
visited.add((0, 1))
print(len(visited))

[(1, 2), (1, 4), (7, 2), (7, 4)]
[(1, 2), (1, 4), (7, 2), (7, 4)]
True False
2


### 3.5. `bisect` và đếm bằng hai con trỏ

`bisect_left(a, x)` / `bisect_right(a, x)` cho vị trí chèn `x` vào mảng **đã sắp xếp** (tìm nhị phân, `O(log n)`): số phần tử `== x` là `bisect_right - bisect_left`. **Hai con trỏ:** với `nums1` tăng, ngưỡng `j` (số phần tử `nums2[j]` còn thỏa `u + nums2[j] <= T`) **giảm dần**, nên đếm số cặp có tổng `<= T` chỉ tốn `O(m + n)` thay vì `O(m · n)`.

In [6]:
a = [1, 2, 2, 2, 5]
print(bisect_left(a, 2), bisect_right(a, 2), "-> có", bisect_right(a, 2) - bisect_left(a, 2), "số 2")


def count_le(nums1, nums2, T):
    """Số cặp (i, j) có nums1[i] + nums2[j] <= T, bằng hai con trỏ."""
    count, j = 0, len(nums2) - 1
    for u in nums1:
        while j >= 0 and u + nums2[j] > T:
            j -= 1                        # ngưỡng j chỉ giảm khi u tăng
        if j < 0:
            break
        count += j + 1
    return count


print([count_le([1, 7, 11], [2, 4, 6], T) for T in (2, 3, 7, 9, 13, 17)])    # [0, 1, 3, 4, 7, 9]

1 4 -> có 3 số 2
[0, 1, 3, 4, 7, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums1 = [1, 7, 11]`, `nums2 = [2, 4, 6]` ta vẽ ma trận tổng (hàng ↔ `nums1[i]`, cột ↔ `nums2[j]`):

| | `2` (j=0) | `4` (j=1) | `6` (j=2) |
|---|---|---|---|
| `1` (i=0) | 3 | 5 | 7 |
| `7` (i=1) | 9 | 11 | 13 |
| `11` (i=2) | 13 | 15 | 17 |

Xếp mọi tổng theo thứ tự tăng: `3, 5, 7, 9, 11, 13, 13, 15, 17`, ứng với các cặp `[1,2],[1,4],[1,6],[7,2],[7,4],[7,6]/[11,2],[11,4],[11,6]`.

**Quan sát then chốt (đếm tay):**

1. Cặp **nhỏ nhất** chắc chắn là ô góc trên-trái `(0, 0) = 3`.
2. Sau khi lấy một ô `(i, j)`, **ứng viên nhỏ nhất kế tiếp** chỉ có thể là **ô ngay bên phải** `(i, j+1)` hoặc **ô ngay bên dưới** `(i+1, j)`, hoặc ô nào đó đã nằm "ở biên" từ trước. Một ô xa hơn luôn **lớn hơn** một ô nằm giữa nó và các ô đã lấy.
3. Do đó ta chỉ cần **một tập nhỏ các ô ở biên** — và luôn lấy ô nhỏ nhất trong biên — đó chính là việc của **min-heap**.

Lấy `k = 4`: biên ban đầu `{(0,0)=3}` → lấy 3, thêm `(0,1)=5` và `(1,0)=9`; lấy 5, thêm `(0,2)=7` (và `(1,1)=11`); lấy 7; biên còn `{9, 11}`; lấy 9. Ra `[1,2],[1,4],[1,6],[7,2]`, đúng bốn cặp nhỏ nhất.

In [7]:
# Tính tay bằng code: sắp xếp toàn bộ ma trận tổng
nums1, nums2 = [1, 7, 11], [2, 4, 6]
cells = sorted((u + v, i, j) for i, u in enumerate(nums1) for j, v in enumerate(nums2))
print("tổng tăng dần:", [c[0] for c in cells])
print("4 cặp đầu:", [[nums1[i], nums2[j]] for _, i, j in cells[:4]])

tổng tăng dần: [3, 5, 7, 9, 11, 13, 13, 15, 17]
4 cặp đầu: [[1, 2], [1, 4], [1, 6], [7, 2]]


## 5. Cách 1 — Brute Force / baseline: sinh mọi cặp rồi sắp xếp

**Ý tưởng.** Tạo **tất cả** `m · n` cặp, sắp xếp theo tổng, lấy `k` cặp đầu. Rất dễ hiểu và là **oracle** tốt cho input nhỏ.

In [8]:
def k_smallest_pairs_brute(nums1, nums2, k):
    pairs = [[u, v] for u in nums1 for v in nums2]        # mọi m * n cặp
    pairs.sort(key=lambda p: p[0] + p[1])                 # sắp xếp theo tổng
    return pairs[:k]


for nums1, nums2, k, expected in examples:
    print(k_smallest_pairs_brute(nums1, nums2, k), "(kỳ vọng", expected, ")")

[[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


**Độ phức tạp (Cách 1).** Gọi `m = len(nums1)`, `n = len(nums2)`. Sinh `m · n` cặp và sắp xếp: **Time `O(m·n·log(m·n))`**, **Space `O(m·n)`**. Với `m = n = 10^5` ta có `10^10` cặp — **không khả thi**.

**Nút thắt:** ta sinh và sắp xếp cả những cặp **chắc chắn không bao giờ cần** (vì `k <= 10^4`). Có cách nào chỉ giữ lại `k` cặp tốt nhất, hoặc chỉ "chạm" tới những cặp cần thiết?

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá: chỉ giữ "biên" các cặp có thể là nhỏ nhất kế tiếp.**

Với ma trận `S[i][j]` tăng theo hàng và cột (mục 3.1): với mỗi hàng `i`, các cặp `(i, 0), (i, 1), (i, 2), ...` **đã sắp xếp tăng dần**. Do đó mỗi hàng là một **dãy đã sắp xếp**, và bài toán trở thành **"hợp nhất `m` dãy đã sắp xếp, lấy `k` phần tử đầu"**.

- Giữ trong **min-heap** **phần tử nhỏ nhất chưa lấy của mỗi hàng**: ban đầu là `(i, 0)` cho mỗi hàng `i`.
- Lấy ra phần tử nhỏ nhất `(i, j)` ⇒ đó là cặp nhỏ nhất kế tiếp. Rồi **đẩy** phần tử kế tiếp cùng hàng `(i, j+1)` vào heap (nếu còn).
- Lặp `k` lần.

Cải tiến thêm: vì ô `(i, 0)` với `i >= k` luôn lớn hơn ít nhất `k` ô `(0..k-1, 0)`, ta chỉ cần xét **`min(k, m)` hàng đầu**. Mỗi lần pop/push tốn `O(log k)` ⇒ tổng **`O(k log k)`**, không phụ thuộc `m · n`.

Hai hướng khác cùng ý "chạm ít cặp": **(a)** bắt đầu từ góc `(0, 0)` và mở rộng sang phải/xuống dưới với `visited`; **(b)** **tìm nhị phân trên giá trị tổng** để biết trước ngưỡng của cặp thứ `k`.

In [9]:
# Mini demo: biên của heap sau mỗi lần pop (nums1 = [1,7,11], nums2 = [2,4,6], k = 4)
nums1, nums2, k = [1, 7, 11], [2, 4, 6], 4
heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
heapq.heapify(heap)
taken = []
while heap and len(taken) < k:
    s, i, j = heapq.heappop(heap)
    taken.append([nums1[i], nums2[j]])
    if j + 1 < len(nums2):
        heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    print(f"lấy {taken[-1]} (tổng {s}); biên còn lại: {sorted(heap)}")

lấy [1, 2] (tổng 3); biên còn lại: [(5, 0, 1), (9, 1, 0), (13, 2, 0)]
lấy [1, 4] (tổng 5); biên còn lại: [(7, 0, 2), (9, 1, 0), (13, 2, 0)]
lấy [1, 6] (tổng 7); biên còn lại: [(9, 1, 0), (13, 2, 0)]
lấy [7, 2] (tổng 9); biên còn lại: [(11, 1, 1), (13, 2, 0)]


## 7. Các cách tiếp theo

Ngoài Cách 1 (sinh mọi cặp), ta có thêm **4 cách khác nhau về ý tưởng**: Cách 2 (max-heap kích thước `k` có cắt tỉa), Cách 3 (min-heap mở rộng theo hàng — cách nộp), Cách 4 (min-heap từ góc `(0, 0)` với `visited`), Cách 5 (tìm nhị phân trên giá trị tổng).

### Cách 2 — Max-heap kích thước `k`: giữ `k` cặp nhỏ nhất đã thấy

**Giải thích từng đoạn.**

1. Duyệt các cặp `(u, v)`; heap `best` giữ **tối đa `k` cặp nhỏ nhất đã thấy**, tổ chức theo **tổng lớn nhất ở đỉnh** (lưu `-tổng`) để biết nhanh "cặp tệ nhất" đang giữ.
2. Nếu heap chưa đầy `k`: `heappush`. Nếu đã đầy và tổng mới **nhỏ hơn** tệ nhất: `heapreplace` (đẩy tệ nhất ra, thêm cặp mới).
3. **Cắt tỉa:** vì `nums2` tăng, khi một tổng `u + v` đã **không nhỏ hơn** tệ nhất (heap đầy) thì mọi `v` sau đó cũng vậy ⇒ `break` vòng trong; tương tự nếu `u + nums2[0]` đã không nhỏ hơn tệ nhất thì mọi `u` sau cũng vậy ⇒ `break` vòng ngoài.
4. Cuối cùng sắp xếp `k` cặp theo tổng tăng.

Đây là cải tiến bộ nhớ so với Cách 1 (`O(k)` thay vì `O(m·n)`), và có cắt tỉa nên thường nhanh hơn rất nhiều — nhưng **trường hợp xấu vẫn có thể duyệt rất nhiều cặp**.

In [10]:
def k_smallest_pairs_maxheap(nums1, nums2, k):
    best = []                                              # max-heap bằng (-tổng, u, v), tối đa k phần tử
    for u in nums1:
        if len(best) == k and u + nums2[0] >= -best[0][0]:
            break                                          # cắt tỉa ngoài: mọi u sau còn lớn hơn
        for v in nums2:
            s = u + v
            if len(best) < k:
                heapq.heappush(best, (-s, u, v))
            elif s < -best[0][0]:
                heapq.heapreplace(best, (-s, u, v))        # bỏ cặp tệ nhất, thêm cặp mới tốt hơn
            else:
                break                                      # cắt tỉa trong: v sau còn lớn hơn
    best.sort(key=lambda t: -t[0])                         # tổng tăng dần
    return [[u, v] for _, u, v in best]


for nums1, nums2, k, expected in examples:
    print(k_smallest_pairs_maxheap(nums1, nums2, k), "(kỳ vọng", expected, ")")

[[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionMaxHeap:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        best = []
        for u in nums1:
            if len(best) == k and u + nums2[0] >= -best[0][0]:
                break
            for v in nums2:
                s = u + v
                if len(best) < k:
                    heapq.heappush(best, (-s, u, v))
                elif s < -best[0][0]:
                    heapq.heapreplace(best, (-s, u, v))
                else:
                    break
        best.sort(key=lambda t: -t[0])
        return [[u, v] for _, u, v in best]


print(SolutionMaxHeap().kSmallestPairs([1, 7, 11], [2, 4, 6], 3))

[[1, 2], [1, 4], [1, 6]]


**Độ phức tạp Cách 2:** xấu nhất `O(m·n·log k)` thời gian (khi cắt tỉa không giúp được), **Space `O(k)`**. Với `m = n = 10^5` và những dữ liệu bất lợi có thể vẫn **quá chậm** — đó là lý do ta cần Cách 3.

### Cách 3 — Min-heap mở rộng theo hàng (cách nộp)

**Giải thích từng đoạn.**

1. Mỗi hàng `i` của ma trận ảo là một dãy tăng: `S[i][0] <= S[i][1] <= ...`. Heap giữ **phần tử nhỏ nhất chưa lấy của mỗi hàng**.
2. **Khởi tạo:** `heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]` — chỉ cần `min(k, m)` hàng đầu (hàng `i >= k` không thể nằm trong `k` cặp nhỏ nhất).
3. **Vòng lặp:** pop `(s, i, j)`, thêm cặp `[nums1[i], nums2[j]]` vào kết quả. Nếu `j + 1 < n`: push `(nums1[i] + nums2[j+1], i, j+1)`.
4. Dừng khi đủ `k` cặp (hoặc heap rỗng).

Mỗi lần pop lấy đúng cặp nhỏ nhất chưa lấy: vì heap luôn chứa phần tử nhỏ nhất của **mỗi hàng**, nên phần tử nhỏ nhất toàn cục chưa lấy chắc chắn nằm trong heap.

In [12]:
def k_smallest_pairs_heap(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]   # phần tử đầu của mỗi hàng
    heapq.heapify(heap)
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)                                # cặp nhỏ nhất chưa lấy
        result.append([nums1[i], nums2[j]])
        if j + 1 < n:                                                # mở rộng sang phải trong cùng hàng
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return result


for nums1, nums2, k, expected in examples:
    print(k_smallest_pairs_heap(nums1, nums2, k), "(kỳ vọng", expected, ")")

[[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionHeap:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        m, n = len(nums1), len(nums2)
        heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]
        heapq.heapify(heap)
        result = []
        while heap and len(result) < k:
            _, i, j = heapq.heappop(heap)
            result.append([nums1[i], nums2[j]])
            if j + 1 < n:
                heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
        return result


print(SolutionHeap().kSmallestPairs([1, 1, 2], [1, 2, 3], 2))

[[1, 1], [1, 1]]


**Độ phức tạp Cách 3:** khởi tạo `O(min(k, m))`, mỗi trong `k` lần lặp có một `pop` và nhiều nhất một `push` trên heap kích thước `<= min(k, m)` ⇒ **Time `O(min(k, m) + k·log min(k, m))`**, **Space `O(min(k, m))`** (không kể kết quả `O(k)`).

### Cách 4 — Min-heap từ góc `(0, 0)` với `visited`

**Giải thích từng đoạn.** Thay vì đưa sẵn mọi hàng, ta **chỉ bắt đầu từ ô nhỏ nhất `(0, 0)`**.

1. `heap = [(nums1[0] + nums2[0], 0, 0)]`, `visited = {(0, 0)}`.
2. Pop `(s, i, j)`, thêm cặp vào kết quả. Hai ô "kế tiếp" có thể là nhỏ nhất: **bên dưới `(i+1, j)`** và **bên phải `(i, j+1)`**.
3. Với mỗi ô kế tiếp hợp lệ (`< m`, `< n`) **chưa thăm**: đánh dấu vào `visited` rồi push. **Cần `visited`** vì một ô như `(i+1, j+1)` có thể được đạt tới từ hai ô khác nhau (`(i+1, j)` và `(i, j+1)`).
4. Dừng khi đủ `k`.

Heap chỉ phình ra tối đa `~k` phần tử và `visited` cũng cỡ `O(k)`, nhưng mỗi lần pop có thể đẩy **hai** ô nên hằng số hơi lớn hơn Cách 3.

In [14]:
def k_smallest_pairs_visited(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    heap = [(nums1[0] + nums2[0], 0, 0)]
    visited = {(0, 0)}
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        for ni, nj in ((i + 1, j), (i, j + 1)):              # ô bên dưới và ô bên phải
            if ni < m and nj < n and (ni, nj) not in visited:
                visited.add((ni, nj))
                heapq.heappush(heap, (nums1[ni] + nums2[nj], ni, nj))
    return result


for nums1, nums2, k, expected in examples:
    print(k_smallest_pairs_visited(nums1, nums2, k), "(kỳ vọng", expected, ")")

[[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionVisited:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        m, n = len(nums1), len(nums2)
        heap = [(nums1[0] + nums2[0], 0, 0)]
        visited = {(0, 0)}
        result = []
        while heap and len(result) < k:
            _, i, j = heapq.heappop(heap)
            result.append([nums1[i], nums2[j]])
            for ni, nj in ((i + 1, j), (i, j + 1)):
                if ni < m and nj < n and (ni, nj) not in visited:
                    visited.add((ni, nj))
                    heapq.heappush(heap, (nums1[ni] + nums2[nj], ni, nj))
        return result


print(SolutionVisited().kSmallestPairs([1, 2], [3], 2))

[[1, 3], [2, 3]]


**Độ phức tạp Cách 4:** mỗi lần pop đẩy tối đa `2` ô; heap và `visited` có `O(k)` phần tử ⇒ **Time `O(k log k)`**, **Space `O(k)`**.

### Cách 5 — Tìm nhị phân trên giá trị tổng

**Giải thích từng đoạn.** Đây là cách **khác hẳn**: thay vì "bóc" từng cặp, ta tìm **tổng của cặp thứ `k`** bằng **tìm nhị phân trên giá trị** rồi thu các cặp có tổng nhỏ hơn hoặc bằng ngưỡng.

1. Hàm `count_le(T)` (mục 3.5) đếm số cặp có tổng `<= T` trong `O(m + n)` bằng hai con trỏ.
2. Tìm **`T` nhỏ nhất** thỏa `count_le(T) >= k` bằng tìm nhị phân trên khoảng `[nums1[0] + nums2[0], nums1[-1] + nums2[-1]]`. `T` chính là **tổng của cặp thứ `k`**.
3. Vì `T` nhỏ nhất như vậy nên `count_le(T - 1) < k`: số cặp có tổng **nhỏ hơn `T`** ít hơn `k`. **Thu tất cả** các cặp đó (ít hơn `k` cặp).
4. Bổ sung cho đủ `k` bằng các cặp có tổng **đúng bằng `T`**: với mỗi `u` tìm các `v == T - u` bằng `bisect_left/right`.
5. Sắp xếp kết quả theo tổng (ổn định).

Cách này **ít bị ảnh hưởng bởi `k`** (chỉ cần `O(k)` cho việc thu thập) nhưng tốn `O((m + n) log R)` cho việc tìm ngưỡng với `R` là khoảng giá trị tổng (`<= 4 · 10^9`, `~32` vòng).

In [16]:
def k_smallest_pairs_bsearch(nums1, nums2, k):
    lo, hi = nums1[0] + nums2[0], nums1[-1] + nums2[-1]
    while lo < hi:                                     # T nhỏ nhất sao cho count_le(T) >= k
        mid = (lo + hi) // 2
        if count_le(nums1, nums2, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    T = lo
    result = []
    for u in nums1:                                    # (1) mọi cặp có tổng < T (ít hơn k cặp)
        if u + nums2[0] >= T:
            break
        for v in nums2:
            if u + v >= T:
                break
            result.append([u, v])
    for u in nums1:                                    # (2) bổ sung cặp có tổng == T cho đủ k
        if len(result) >= k:
            break
        need = T - u
        left, right = bisect_left(nums2, need), bisect_right(nums2, need)
        for _ in range(min(right - left, k - len(result))):
            result.append([u, need])
    result.sort(key=lambda p: p[0] + p[1])
    return result


for nums1, nums2, k, expected in examples:
    print(k_smallest_pairs_bsearch(nums1, nums2, k), "(kỳ vọng", expected, ")")

[[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionBinarySearch:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        def count_le(T):
            count, j = 0, len(nums2) - 1
            for u in nums1:
                while j >= 0 and u + nums2[j] > T:
                    j -= 1
                if j < 0:
                    break
                count += j + 1
            return count

        lo, hi = nums1[0] + nums2[0], nums1[-1] + nums2[-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if count_le(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        T = lo
        result = []
        for u in nums1:
            if u + nums2[0] >= T:
                break
            for v in nums2:
                if u + v >= T:
                    break
                result.append([u, v])
        for u in nums1:
            if len(result) >= k:
                break
            need = T - u
            left, right = bisect_left(nums2, need), bisect_right(nums2, need)
            for _ in range(min(right - left, k - len(result))):
                result.append([u, need])
        result.sort(key=lambda p: p[0] + p[1])
        return result


print(SolutionBinarySearch().kSmallestPairs([1, 7, 11], [2, 4, 6], 5))

[[1, 2], [1, 4], [1, 6], [7, 2], [7, 4]]


**Độ phức tạp Cách 5:** tìm ngưỡng `O((m + n) · log R)`; thu cặp `< T` tốn `O(m + k)`; bổ sung `O(m log n + k)`; sắp xếp `O(k log k)` ⇒ **Time `O((m + n) log R + m log n + k log k)`**, **Space `O(k)`**.

## 8. TRACE từng bước

Ta trace **Cách 3** (min-heap theo hàng) và **Cách 4** (từ góc với `visited`) trên `nums1 = [1, 7, 11]`, `nums2 = [2, 4, 6]`, `k = 5`, rồi **Cách 5** (tìm nhị phân) trên `k = 5`.

Bảng trace Cách 3 (`k = 5`, ban đầu heap có `(3,0,0)`, `(9,1,0)`, `(13,2,0)`):

| Lần | pop `(tổng, i, j)` | cặp lấy | push mới | heap sau đó (theo tổng) |
|---|---|---|---|---|
| 1 | `(3,0,0)` | `[1,2]` | `(5,0,1)` | `5, 9, 13` |
| 2 | `(5,0,1)` | `[1,4]` | `(7,0,2)` | `7, 9, 13` |
| 3 | `(7,0,2)` | `[1,6]` | không (`j+1 = 3` hết hàng) | `9, 13` |
| 4 | `(9,1,0)` | `[7,2]` | `(11,1,1)` | `11, 13` |
| 5 | `(11,1,1)` | `[7,4]` | `(13,1,2)` | `13, 13` |

Kết quả: `[[1,2],[1,4],[1,6],[7,2],[7,4]]` (tổng `3, 5, 7, 9, 11`).

In [18]:
def trace_heap(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]
    heapq.heapify(heap)
    print(f"khởi tạo heap (phần tử đầu mỗi hàng): {sorted(heap)}")
    result = []
    step = 0
    while heap and len(result) < k:
        step += 1
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        pushed = None
        if j + 1 < n:
            pushed = (nums1[i] + nums2[j + 1], i, j + 1)
            heapq.heappush(heap, pushed)
        print(f"lần {step}: pop (tổng={s}, i={i}, j={j}) -> cặp {result[-1]}; push {pushed}; heap = {sorted(heap)}")
    return result


assert trace_heap([1, 7, 11], [2, 4, 6], 5) == [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4]]

khởi tạo heap (phần tử đầu mỗi hàng): [(3, 0, 0), (9, 1, 0), (13, 2, 0)]
lần 1: pop (tổng=3, i=0, j=0) -> cặp [1, 2]; push (5, 0, 1); heap = [(5, 0, 1), (9, 1, 0), (13, 2, 0)]
lần 2: pop (tổng=5, i=0, j=1) -> cặp [1, 4]; push (7, 0, 2); heap = [(7, 0, 2), (9, 1, 0), (13, 2, 0)]
lần 3: pop (tổng=7, i=0, j=2) -> cặp [1, 6]; push None; heap = [(9, 1, 0), (13, 2, 0)]
lần 4: pop (tổng=9, i=1, j=0) -> cặp [7, 2]; push (11, 1, 1); heap = [(11, 1, 1), (13, 2, 0)]
lần 5: pop (tổng=11, i=1, j=1) -> cặp [7, 4]; push (13, 1, 2); heap = [(13, 1, 2), (13, 2, 0)]


In [19]:
def trace_visited(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    heap = [(nums1[0] + nums2[0], 0, 0)]
    visited = {(0, 0)}
    result = []
    step = 0
    while heap and len(result) < k:
        step += 1
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        added = []
        for ni, nj in ((i + 1, j), (i, j + 1)):
            if ni < m and nj < n and (ni, nj) not in visited:
                visited.add((ni, nj))
                heapq.heappush(heap, (nums1[ni] + nums2[nj], ni, nj))
                added.append((ni, nj))
        print(f"lần {step}: pop (i={i}, j={j}, tổng={s}) -> {result[-1]}; thêm ô {added}; heap = {sorted(heap)}")
    return result


assert trace_visited([1, 7, 11], [2, 4, 6], 5) == [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4]]

lần 1: pop (i=0, j=0, tổng=3) -> [1, 2]; thêm ô [(1, 0), (0, 1)]; heap = [(5, 0, 1), (9, 1, 0)]
lần 2: pop (i=0, j=1, tổng=5) -> [1, 4]; thêm ô [(1, 1), (0, 2)]; heap = [(7, 0, 2), (9, 1, 0), (11, 1, 1)]
lần 3: pop (i=0, j=2, tổng=7) -> [1, 6]; thêm ô [(1, 2)]; heap = [(9, 1, 0), (11, 1, 1), (13, 1, 2)]
lần 4: pop (i=1, j=0, tổng=9) -> [7, 2]; thêm ô [(2, 0)]; heap = [(11, 1, 1), (13, 1, 2), (13, 2, 0)]
lần 5: pop (i=1, j=1, tổng=11) -> [7, 4]; thêm ô [(2, 1)]; heap = [(13, 1, 2), (13, 2, 0), (15, 2, 1)]


In [20]:
def trace_bsearch(nums1, nums2, k):
    lo, hi = nums1[0] + nums2[0], nums1[-1] + nums2[-1]
    print(f"tìm T trong [{lo}, {hi}] sao cho count_le(T) >= {k}")
    while lo < hi:
        mid = (lo + hi) // 2
        c = count_le(nums1, nums2, mid)
        print(f"  lo={lo}, hi={hi}, mid={mid}: count_le(mid) = {c} {'>=' if c >= k else '<'} {k}")
        if c >= k:
            hi = mid
        else:
            lo = mid + 1
    T = lo
    print(f"T = {T}: count_le(T-1) = {count_le(nums1, nums2, T - 1)} < {k} <= count_le(T) = {count_le(nums1, nums2, T)}")
    ans = k_smallest_pairs_bsearch(nums1, nums2, k)
    print("kết quả:", ans, "| tổng:", [p[0] + p[1] for p in ans])
    return ans


assert [p[0] + p[1] for p in trace_bsearch([1, 7, 11], [2, 4, 6], 5)] == [3, 5, 7, 9, 11]

tìm T trong [3, 17] sao cho count_le(T) >= 5
  lo=3, hi=17, mid=10: count_le(mid) = 4 < 5
  lo=11, hi=17, mid=14: count_le(mid) = 7 >= 5
  lo=11, hi=14, mid=12: count_le(mid) = 5 >= 5
  lo=11, hi=12, mid=11: count_le(mid) = 5 >= 5
T = 11: count_le(T-1) = 4 < 5 <= count_le(T) = 5
kết quả: [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4]] | tổng: [3, 5, 7, 9, 11]


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (min-heap theo hàng): `O(k log min(k, m))`, ngắn và an toàn cả khi `m, n = 10^5`.

In [21]:
class Solution:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        m, n = len(nums1), len(nums2)
        # mỗi hàng i bắt đầu bằng cặp (nums1[i], nums2[0]); chỉ cần min(k, m) hàng đầu
        heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]
        heapq.heapify(heap)
        result = []
        while heap and len(result) < k:
            _, i, j = heapq.heappop(heap)                  # cặp nhỏ nhất chưa lấy
            result.append([nums1[i], nums2[j]])
            if j + 1 < n:                                  # cặp kế tiếp của cùng hàng
                heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
        return result


sol = Solution()
for nums1, nums2, k, expected in examples:
    print(nums1, nums2, k, "->", sol.kSmallestPairs(nums1, nums2, k), "(kỳ vọng", expected, ")")

[1, 7, 11] [2, 4, 6] 3 -> [[1, 2], [1, 4], [1, 6]] (kỳ vọng [[1, 2], [1, 4], [1, 6]] )
[1, 1, 2] [1, 2, 3] 2 -> [[1, 1], [1, 1]] (kỳ vọng [[1, 1], [1, 1]] )
[1, 2] [3] 2 -> [[1, 3], [2, 3]] (kỳ vọng [[1, 3], [2, 3]] )


## 10. Vì sao thuật toán đúng

- **Mỗi hàng là một dãy tăng.** Với hàng `i` cố định, `S[i][0] <= S[i][1] <= ... <= S[i][n-1]` (vì `nums2` tăng). Cặp `(i, j)` chỉ có thể được lấy **sau** cặp `(i, j-1)`, và heap chỉ chứa `(i, j)` sau khi `(i, j-1)` đã được lấy.
- **Bất biến.** Tại mọi thời điểm, heap chứa đúng **phần tử nhỏ nhất chưa lấy** của mỗi hàng (trong `min(k, m)` hàng đầu). Do đó phần tử nhỏ nhất của heap là **phần tử nhỏ nhất chưa lấy của toàn bộ các cặp đó**: nếu có cặp chưa lấy nhỏ hơn, nó thuộc một hàng nào đó và phần tử nhỏ nhất của hàng đó (có trong heap) sẽ còn nhỏ hơn hoặc bằng nó — mâu thuẫn.
- **Các hàng `i >= k` không cần xét.** `k` ô `(0, 0), (1, 0), ..., (k-1, 0)` đều `<= S[i][j]` với mọi `i >= k` (vì `nums1` tăng, `nums2` tăng). Nên có ít nhất `k` cặp **không lớn hơn** bất kỳ cặp nào thuộc hàng `i >= k`; những cặp đó không thể nằm duy nhất trong top `k` (khi bằng nhau, đáp án vẫn hợp lệ).
- **Dừng.** Sau `k` lần pop ta có `k` cặp nhỏ nhất theo thứ tự tăng dần (hoặc hết cặp, nhưng ràng buộc `k <= m·n` đảm bảo đủ).
- **Nhiều đáp án đúng.** Khi nhiều cặp cùng tổng, bất kỳ cách chọn nào miễn đủ `k` cặp có tổng nhỏ nhất đều hợp lệ — vì vậy khi test ta so **dãy tổng** và kiểm tra **cặp có tồn tại** (mục 12).

## 11. Phân tích Time / Space Complexity

Gọi `m = len(nums1)`, `n = len(nums2)`, `R` là khoảng giá trị tổng (`<= 4·10^9`).

| Cách | Time | Space |
|---|---|---|
| 1. Sinh mọi cặp + sắp xếp | `O(m·n·log(m·n))` | `O(m·n)` |
| 2. Max-heap kích thước `k` (cắt tỉa) | xấu nhất `O(m·n·log k)` | `O(k)` |
| 3. Min-heap mở rộng theo hàng | `O(k log min(k, m))` | `O(min(k, m))` |
| 4. Min-heap từ góc + `visited` | `O(k log k)` | `O(k)` |
| 5. Tìm nhị phân trên tổng | `O((m+n) log R + m log n + k log k)` | `O(k)` |

Với `m = n = 10^5`, `k = 10^4`: Cách 1 cần `10^10` cặp (không khả thi); Cách 3 chỉ cỡ `10^4 · 14 ~ 1.4·10^5` phép heap — tức thì.

## 12. Bộ test

Bài này có **nhiều đáp án đúng** (cặp cùng tổng có thể xếp khác nhau), nên ta **không** so sánh bằng `==`. Ta viết **hàm kiểm tra tính hợp lệ** `is_valid_answer(nums1, nums2, k, result)`:

1. `result` có **đúng `k` phần tử**, mỗi phần tử là một cặp `[u, v]`.
2. Các tổng `u + v` **không giảm** (kết quả đã sắp xếp).
3. Mỗi cặp `(u, v)` **thực sự có thể tạo** từ hai mảng, **kể cả bội số** (cặp `[1,1]` xuất hiện tối đa `cnt1[1] · cnt2[1]` lần).
4. **Dãy tổng** bằng đúng **dãy `k` tổng nhỏ nhất** — tính bằng một **oracle độc lập** `oracle_sums` dựa trên `heapq.merge` (hợp nhất `min(k, m)` dãy đã sắp xếp, cách làm khác hẳn các cách của ta).

In [22]:
def oracle_sums(nums1, nums2, k):
    """k tổng nhỏ nhất: hợp nhất các hàng bằng heapq.merge (độc lập với các cách ở trên)."""
    def row(u):                                                  # hàm riêng để mỗi dãy "nhớ" đúng u của nó
        return (u + v for v in nums2)

    rows = [row(u) for u in nums1[:k]]                           # hàng i >= k không thể góp mặt
    return list(itertools.islice(heapq.merge(*rows), k))


def is_valid_answer(nums1, nums2, k, result):
    if len(result) != k or any(len(p) != 2 for p in result):
        return False
    sums = [p[0] + p[1] for p in result]
    if any(sums[i] > sums[i + 1] for i in range(len(sums) - 1)):      # tổng không giảm
        return False
    c1, c2 = Counter(nums1), Counter(nums2)
    for (u, v), times in Counter(map(tuple, result)).items():          # cặp phải tồn tại, kể cả bội số
        if times > c1[u] * c2[v]:
            return False
    return sums == oracle_sums(nums1, nums2, k)                        # đúng k tổng nhỏ nhất


sol = Solution()
impls = {
    "brute": k_smallest_pairs_brute,
    "maxheap": k_smallest_pairs_maxheap,
    "heap": k_smallest_pairs_heap,
    "visited": k_smallest_pairs_visited,
    "bsearch": k_smallest_pairs_bsearch,
    "leetcode": sol.kSmallestPairs,
}


def check_all(nums1, nums2, k, label, keys=None):
    for key, fn in impls.items():
        if keys is not None and key not in keys:
            continue
        result = fn(list(nums1), list(nums2), k)
        assert is_valid_answer(nums1, nums2, k, result), (label, key, nums1, nums2, k, result)
    print("PASS", label)


for idx, (nums1, nums2, k, expected) in enumerate(examples, 1):
    check_all(nums1, nums2, k, f"ví dụ {idx}")
    assert is_valid_answer(nums1, nums2, k, expected), "đáp án mẫu của đề cũng phải hợp lệ"

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [23]:
check_all([1], [1], 1, "một phần tử mỗi mảng")
check_all([1, 2], [3], 2, "nums2 một phần tử, k = m·n")
check_all([3], [1, 2, 5], 3, "nums1 một phần tử, k = m·n")
check_all([1, 1, 1], [1, 1, 1], 9, "toàn giá trị bằng nhau, lấy hết")
check_all([1, 1, 1], [1, 1, 1], 4, "toàn giá trị bằng nhau, lấy 4")
check_all([-5, -3, 0, 4], [-4, -4, 2], 7, "có số âm và trùng lặp")
check_all([-10 ** 9, 10 ** 9], [-10 ** 9, 10 ** 9], 4, "giá trị biên ±10^9, lấy hết")
check_all([-10 ** 9, 10 ** 9], [-10 ** 9, 10 ** 9], 3, "giá trị biên ±10^9, lấy 3")
check_all([1, 2, 3], [1, 2, 3], 9, "k = m·n (lấy toàn bộ)")
check_all(list(range(1, 6)), [1], 5, "một cột, k = m")
check_all([1], list(range(1, 6)), 5, "một hàng, k = n")
check_all([0, 0, 0, 1], [0, 0, 5], 6, "nhiều cặp trùng tổng ở biên k")

PASS một phần tử mỗi mảng
PASS nums2 một phần tử, k = m·n
PASS nums1 một phần tử, k = m·n
PASS toàn giá trị bằng nhau, lấy hết
PASS toàn giá trị bằng nhau, lấy 4
PASS có số âm và trùng lặp
PASS giá trị biên ±10^9, lấy hết
PASS giá trị biên ±10^9, lấy 3
PASS k = m·n (lấy toàn bộ)
PASS một cột, k = m
PASS một hàng, k = n
PASS nhiều cặp trùng tổng ở biên k


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi hay gặp: (a) **chỉ mở rộng sang phải từ `(0, 0)`** (bỏ sót hàng dưới); (b) **mở rộng cả hai hướng nhưng không `visited`** (trùng cặp); (c) **chỉ ghép `nums1[0]` với `k` phần tử đầu của `nums2`** (bỏ sót mọi hàng khác). Hàm kiểm tra `is_valid_answer` phải **bắt được** chúng.

In [24]:
def wrong_only_right(nums1, nums2, k):
    heap = [(nums1[0] + nums2[0], 0, 0)]
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):                             # SAI: không bao giờ chuyển sang hàng dưới
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return result


def wrong_no_visited(nums1, nums2, k):
    heap = [(nums1[0] + nums2[0], 0, 0)]
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        if i + 1 < len(nums1):
            heapq.heappush(heap, (nums1[i + 1] + nums2[j], i + 1, j))
        if j + 1 < len(nums2):                             # SAI: ô (i+1, j+1) bị đẩy hai lần
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return result


def wrong_first_k_only(nums1, nums2, k):
    pairs = [[u, v] for u in nums1[:1] for v in nums2[:k]]    # SAI: chỉ xét hàng đầu
    return pairs[:k]


tests = [([1, 7, 11], [2, 4, 6], 4), ([1, 1, 2], [1, 2, 3], 5), ([1, 2], [100, 200], 3)]
for fn in (wrong_only_right, wrong_no_visited, wrong_first_k_only):
    results = [is_valid_answer(a, b, k, fn(a, b, k)) for a, b, k in tests]
    print(fn.__name__, "hợp lệ trên từng test:", results)
    assert not all(results), fn.__name__ + " phải sai ở ít nhất một test"

wrong_only_right hợp lệ trên từng test: [False, False, False]
wrong_no_visited hợp lệ trên từng test: [True, False, True]
wrong_first_k_only hợp lệ trên từng test: [False, False, False]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Đầu vào hợp lệ theo ràng buộc: hai mảng **không giảm**, độ dài `1..10^5`, giá trị `-10^9..10^9`, `1 <= k <= min(10^4, m·n)`. Ta kiểm tra:

1. **Nhiều test nhỏ** (độ dài `1..8`, giá trị `-5..5` để có nhiều trùng lặp, mọi `k` từ `1` đến `m·n`): **cả 6 hàm**.
2. **Test vừa** (độ dài tới `200`, giá trị tới `±10^9`, `k` tới `3000`): **cả 6 hàm**.
3. **Test lớn** (`m = n = 10^5`, giá trị tới `±10^9`, `k = 10^4`): chỉ các cách không sinh `m·n` cặp — Cách 3, Cách 4, Cách 5 và bản nộp.

In [25]:
rng = random.Random(373)


def random_sorted(n, lo, hi):
    return sorted(rng.randint(lo, hi) for _ in range(n))


count = 0
for trial in range(1500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    nums1, nums2 = random_sorted(m, -5, 5), random_sorted(n, -5, 5)
    k = rng.randint(1, m * n)
    for key, fn in impls.items():
        assert is_valid_answer(nums1, nums2, k, fn(list(nums1), list(nums2), k)), (key, nums1, nums2, k)
    count += 1
print("PASS", count, "test nhỏ ngẫu nhiên (6 hàm)")

for m, n in [(1, 1), (2, 3), (3, 2), (4, 4), (5, 3)]:             # vét cạn mọi k cho vài cặp độ dài
    nums1, nums2 = random_sorted(m, -3, 3), random_sorted(n, -3, 3)
    for k in range(1, m * n + 1):
        for key, fn in impls.items():
            assert is_valid_answer(nums1, nums2, k, fn(list(nums1), list(nums2), k)), (key, nums1, nums2, k)
print("PASS vét cạn mọi k cho vài cặp độ dài")

for trial in range(40):
    m, n = rng.randint(1, 200), rng.randint(1, 200)
    nums1, nums2 = random_sorted(m, -10 ** 9, 10 ** 9), random_sorted(n, -10 ** 9, 10 ** 9)
    k = rng.randint(1, min(3000, m * n))
    for key, fn in impls.items():
        assert is_valid_answer(nums1, nums2, k, fn(list(nums1), list(nums2), k)), (key, m, n, k)
print("PASS 40 test vừa ngẫu nhiên (6 hàm)")

PASS 1500 test nhỏ ngẫu nhiên (6 hàm)
PASS vét cạn mọi k cho vài cặp độ dài


PASS 40 test vừa ngẫu nhiên (6 hàm)


In [26]:
# Test lớn: m = n = 10^5, k = 10^4 (không dùng Cách 1, 2)
big1 = random_sorted(100_000, -10 ** 9, 10 ** 9)
big2 = random_sorted(100_000, -10 ** 9, 10 ** 9)
for key in ("heap", "visited", "bsearch", "leetcode"):
    result = impls[key](list(big1), list(big2), 10_000)
    assert is_valid_answer(big1, big2, 10_000, result), key
print("PASS test lớn 10^5 x 10^5, k = 10^4 (Cách 3, 4, 5, bản nộp)")

# Test lớn có nhiều giá trị trùng (khó cho cắt tỉa): toàn số bằng nhau
same = [7] * 100_000
for key in ("heap", "visited", "bsearch", "leetcode"):
    result = impls[key](same, same, 10_000)
    assert is_valid_answer(same, same, 10_000, result), key
print("PASS test lớn toàn giá trị bằng nhau")

PASS test lớn 10^5 x 10^5, k = 10^4 (Cách 3, 4, 5, bản nộp)
PASS test lớn toàn giá trị bằng nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đẩy mọi cặp vào heap

In [27]:
def loi1_day_het(nums1, nums2, k):
    heap = []
    for u in nums1:
        for v in nums2:                              # SAI: m * n phần tử trong heap
            heapq.heappush(heap, (u + v, u, v))
    return [[u, v] for _, u, v in (heapq.heappop(heap) for _ in range(k))]


nums1, nums2 = [1, 7, 11], [2, 4, 6]
print("kết quả:", loi1_day_het(nums1, nums2, 3), "| số phần tử đã push:", len(nums1) * len(nums2))
print("với m = n = 10^5 sẽ cần", 10 ** 5 * 10 ** 5, "phần tử trong heap")

kết quả: [[1, 2], [1, 4], [1, 6]] | số phần tử đã push: 9
với m = n = 10^5 sẽ cần 10000000000 phần tử trong heap


**Vì sao sai:** đúng kết quả với input nhỏ, nhưng tốn `O(m·n)` thời gian và bộ nhớ — `10^10` phần tử khi `m = n = 10^5` (hết bộ nhớ, TLE). **Cách sửa:** chỉ giữ **biên** trong heap (Cách 3 / 4).

### Lỗi 2: Quên kiểm tra chỉ số khi mở rộng (`IndexError`)

In [28]:
def loi2_tran_chi_so(nums1, nums2, k):
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
    heapq.heapify(heap)
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))     # SAI: j + 1 có thể bằng n
    return result


try:
    loi2_tran_chi_so([1, 2], [3], 2)
except IndexError as err:
    print("IndexError:", err)
print("đúng:", Solution().kSmallestPairs([1, 2], [3], 2))

IndexError: list index out of range
đúng: [[1, 3], [2, 3]]


**Vì sao sai:** khi `j + 1 == len(nums2)` thì `nums2[j + 1]` ngoài phạm vi. **Cách sửa:** `if j + 1 < n:` trước khi `push` (và `ni < m`, `nj < n` ở Cách 4).

### Lỗi 3: Hai hướng mở rộng nhưng không `visited` (trùng cặp)

In [29]:
a, b, k = [1, 7, 11], [2, 4, 6], 9
bad = wrong_no_visited(a, b, k)
print("kết quả sai:", bad)
print("có trùng cặp?", len(bad) != len({tuple(p) for p in bad}), "| hợp lệ?", is_valid_answer(a, b, k, bad))
print("đúng:", Solution().kSmallestPairs(a, b, k))

kết quả sai: [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4], [7, 4], [7, 6], [7, 6], [7, 6]]
có trùng cặp? True | hợp lệ? False
đúng: [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4], [7, 6], [11, 2], [11, 4], [11, 6]]


**Vì sao sai:** ô `(i+1, j+1)` được đạt tới từ **cả hai** ô `(i+1, j)` và `(i, j+1)`, nên bị push hai lần ⇒ cùng một cặp xuất hiện hai lần trong kết quả. **Cách sửa:** `visited` (Cách 4), hoặc chỉ mở rộng **một** hướng như Cách 3 (hàng đã được khởi tạo sẵn, ta chỉ đi sang phải).

### Lỗi 4: Chỉ đi sang phải từ `(0, 0)` (bỏ sót các hàng dưới)

In [30]:
a, b, k = [1, 2], [100, 200], 3
print("sai:", wrong_only_right(a, b, k), "| hợp lệ?", is_valid_answer(a, b, k, wrong_only_right(a, b, k)))
print("đúng:", Solution().kSmallestPairs(a, b, k))

sai: [[1, 100], [1, 200]] | hợp lệ? False
đúng: [[1, 100], [2, 100], [1, 200]]


**Vì sao sai:** với `nums1 = [1, 2]`, `nums2 = [100, 200]` thì cặp `[2, 100]` (tổng `102`) nhỏ hơn `[1, 200]` (tổng `201`), nhưng code chỉ đi sang phải nên bỏ sót cặp thuộc hàng `i = 1`. **Cách sửa:** khởi tạo **mọi hàng** (Cách 3) hoặc mở rộng cả **xuống dưới** (Cách 4).

### Lỗi 5: Dùng `==` để so với đáp án mẫu khi có nhiều đáp án đúng

In [31]:
a, b, k = [1, 7, 11], [2, 4, 6], 7                 # tổng 13 xuất hiện hai lần: [7,6] và [11,2]
mine = Solution().kSmallestPairs(a, b, k)
doc_order = [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4], [11, 2], [7, 6]]   # thứ tự trong mô tả đề
print("của mình :", mine)
print("trong đề :", doc_order)
print("== nhau?", mine == doc_order, "| cả hai đều hợp lệ?", is_valid_answer(a, b, k, mine), is_valid_answer(a, b, k, doc_order))

của mình : [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4], [7, 6], [11, 2]]
trong đề : [[1, 2], [1, 4], [1, 6], [7, 2], [7, 4], [11, 2], [7, 6]]
== nhau? False | cả hai đều hợp lệ? True True


**Vì sao sai:** khi nhiều cặp cùng tổng (ở đây `[7,6]` và `[11,2]` đều `13`), mọi thứ tự đều hợp lệ; so sánh `==` với một thứ tự cụ thể sẽ báo sai oan. **Cách sửa:** viết **hàm kiểm tra hợp lệ** (đủ `k` cặp, tồn tại, tổng đúng `k` tổng nhỏ nhất) như `is_valid_answer`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**`k` phần tử nhỏ nhất từ nhiều dãy đã sắp xếp**" (hoặc từ ma trận có hàng/cột tăng) ⇒ **min-heap** chứa **phần tử đầu của mỗi dãy**, pop rồi push phần tử kế tiếp cùng dãy.
- Chỉ cần khởi tạo **`min(k, m)` hàng**: hàng nào có phần tử đầu không thể thuộc top `k` thì khỏi đưa vào.
- Heap chứa tuple `(giá_trị, i, j)`: Python so sánh từ trái sang phải; có thể không cần `visited` nếu mỗi ô chỉ có **một** đường đến nó (khởi tạo mọi hàng, chỉ đi sang phải).
- Nếu cần **chỉ giá trị thứ `k`** (không cần cặp): **tìm nhị phân trên giá trị** + đếm bằng hai con trỏ (`O((m+n) log R)`).
- Max-heap bằng cách lưu `-giá_trị`; `heapreplace` hiệu quả cho "giữ `k` nhỏ nhất".
- Khi có **nhiều đáp án đúng**, test bằng **hàm kiểm tra hợp lệ** và **oracle độc lập** (ở đây là `heapq.merge`).
- Khi nghe `10^5 x 10^5` mà `k <= 10^4`, tức là không được sinh `m·n` cặp: phải "chạm" tối đa khoảng `k` ô.

## 15. Biến thể của bài

1. **Kth Smallest Element in a Sorted Matrix (378)** — cùng ý (hàng/cột tăng).
2. **Merge k Sorted Lists (23)** — heap chứa nút đầu của mỗi danh sách.
3. **Find K Pairs with Largest Sums** — đảo dấu hoặc duyệt từ cuối.
4. **K-th Smallest Prime Fraction (786)** — heap hoặc tìm nhị phân trên giá trị phân số.
5. **Ugly Number II (264)**, **Super Ugly Number (313)** — heap / con trỏ trên nhiều dãy tăng.
6. **Kth smallest sum of three arrays** — mở rộng 3 chiều với `visited`.

In [32]:
def k_largest_pairs(nums1, nums2, k):
    """k cặp có tổng lớn nhất: dùng đối xứng (đảo dấu rồi lấy nhỏ nhất)."""
    neg1 = sorted(-x for x in nums1)
    neg2 = sorted(-x for x in nums2)
    return [[-u, -v] for u, v in Solution().kSmallestPairs(neg1, neg2, k)]


def kth_smallest_pair_sum(nums1, nums2, k):
    """Chỉ giá trị tổng thứ k (tìm nhị phân trên giá trị)."""
    lo, hi = nums1[0] + nums2[0], nums1[-1] + nums2[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(nums1, nums2, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


def k_smallest_triples_sums(nums1, nums2, nums3, k):
    """k tổng nhỏ nhất của bộ ba (một phần tử mỗi mảng): heap + visited trên (i, j, l)."""
    heap = [(nums1[0] + nums2[0] + nums3[0], 0, 0, 0)]
    visited = {(0, 0, 0)}
    sums = []
    while heap and len(sums) < k:
        s, i, j, l = heapq.heappop(heap)
        sums.append(s)
        for ni, nj, nl in ((i + 1, j, l), (i, j + 1, l), (i, j, l + 1)):
            if ni < len(nums1) and nj < len(nums2) and nl < len(nums3) and (ni, nj, nl) not in visited:
                visited.add((ni, nj, nl))
                heapq.heappush(heap, (nums1[ni] + nums2[nj] + nums3[nl], ni, nj, nl))
    return sums


print(k_largest_pairs([1, 7, 11], [2, 4, 6], 3))                  # tổng lớn nhất: 17, 15, 13
print(kth_smallest_pair_sum([1, 7, 11], [2, 4, 6], 5))             # 11
print(k_smallest_triples_sums([1, 2], [1, 2], [1, 2], 5))          # [3, 4, 4, 4, 5]
rng3 = random.Random(7)
for _ in range(300):
    a = sorted(rng3.randint(-6, 6) for _ in range(rng3.randint(1, 5)))
    b = sorted(rng3.randint(-6, 6) for _ in range(rng3.randint(1, 5)))
    c = sorted(rng3.randint(-6, 6) for _ in range(rng3.randint(1, 4)))
    k = rng3.randint(1, len(a) * len(b))
    assert kth_smallest_pair_sum(a, b, k) == sorted(x + y for x in a for y in b)[k - 1]
    kk = rng3.randint(1, len(a) * len(b) * len(c))
    assert k_smallest_triples_sums(a, b, c, kk) == sorted(x + y + z for x in a for y in b for z in c)[:kk]
    top = k_largest_pairs(a, b, k)
    assert [p[0] + p[1] for p in top] == sorted((x + y for x in a for y in b), reverse=True)[:k]
print("các hàm biến thể đều đúng")

[[11, 6], [11, 4], [11, 2]]
11
[3, 4, 4, 4, 5]
các hàm biến thể đều đúng


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh mọi cặp | sắp xếp `m·n` cặp | `O(m·n·log(m·n))` | `O(m·n)` | input rất nhỏ; làm **oracle** |
| 2. Max-heap `k` + cắt tỉa | giữ `k` cặp nhỏ nhất đã thấy | xấu nhất `O(m·n·log k)` | `O(k)` | nhanh khi dữ liệu thuận lợi, cài đơn giản |
| 3. Min-heap theo hàng | heap chứa phần tử đầu mỗi hàng | `O(k log min(k, m))` | `O(min(k, m))` | **nên nộp**: ngắn, chắc chắn |
| 4. Min-heap từ góc + `visited` | mở rộng xuống/phải với tập đã thăm | `O(k log k)` | `O(k)` | ma trận tổng quát, nhiều chiều |
| 5. Tìm nhị phân trên tổng | tìm ngưỡng `T` của cặp thứ `k` rồi thu thập | `O((m+n) log R + k log k)` | `O(k)` | `k` rất lớn hoặc chỉ cần giá trị thứ `k` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: sau `heappush` lần lượt `5, 1, 3`, `heap[0]` là gì? `sorted([(3, 1), (2, 9), (3, 0)])`, `min(5, 3)`, và `Solution().kSmallestPairs([1, 2], [3], 2)`.

**Bài 2 (tính tay).** Với `nums1 = [1, 2]`, `nums2 = [3, 4]`, `k = 3`: vẽ ma trận tổng `2 x 2`, rồi liệt kê `3` cặp nhỏ nhất (lưu ý có hai cặp cùng tổng `5`).

**Bài 3 (điền TODO).** Hoàn thành `k_smallest_pairs_todo(nums1, nums2, k)` theo Cách 3.

In [33]:
def k_smallest_pairs_todo(nums1, nums2, k):
    # TODO 1: heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]; heapq.heapify(heap)
    # TODO 2: result = []; while heap and len(result) < k: pop (s, i, j) và thêm [nums1[i], nums2[j]] vào result
    # TODO 3: nếu j + 1 < len(nums2): push (nums1[i] + nums2[j + 1], i, j + 1)
    # TODO 4: trả về result
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `k_smallest_pairs_visited2(nums1, nums2, k)` (Cách 4) **không nhìn code mẫu**: heap bắt đầu từ `(0, 0)`, mở rộng `(i+1, j)` và `(i, j+1)` với tập `visited`. Kiểm tra bằng `is_valid_answer` trên các ví dụ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kth_smallest_in_sorted_matrix(matrix, k)` cho **LeetCode 378** (ma trận `n x n` mà hàng và cột tăng): giá trị nhỏ thứ `k`. (b) Viết `k_pairs_with_sum_at_most(nums1, nums2, T)` trả về **số cặp** có tổng `<= T` bằng hai con trỏ (`O(m + n)`) và kiểm tra bằng vét cạn. (c) Khi nào Cách 5 tốt hơn Cách 3? (gợi ý: so sánh `k log k` với `(m + n) log R`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`; `[(2, 9), (3, 0), (3, 1)]`; `3`; `[[1, 3], [2, 3]]`. Bài 3: chép lại các bước ở Cách 3; nhớ điều kiện `j + 1 < len(nums2)`.
- **Mức 2 (vừa):** Bài 2: ma trận `[[4, 5], [5, 6]]`; ba cặp nhỏ nhất: `[1,3]` (4), rồi `[1,4]` và `[2,3]` (cùng tổng 5) theo thứ tự nào cũng hợp lệ. Bài 4: mỗi lần pop, thử `(i+1, j)` và `(i, j+1)`; nhớ `visited.add` **trước khi** push.
- **Mức 3 (sâu):** Bài 5(a): bắt đầu heap bằng phần tử đầu mỗi hàng (hoặc dùng Cách 4); pop `k - 1` lần; phần tử pop thứ `k` là đáp án. 5(b): giống `count_le`. 5(c): Cách 5 hợp khi chỉ cần **giá trị** tổng thứ `k` (không phải liệt kê cặp), hoặc khi `k` rất lớn (ví dụ `10^9`) khiến `k log k` của heap quá nặng, vì khi đó chi phí `(m + n) log R` không phụ thuộc `k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
h = []
for x in (5, 1, 3):
    heapq.heappush(h, x)
assert h[0] == 1
assert sorted([(3, 1), (2, 9), (3, 0)]) == [(2, 9), (3, 0), (3, 1)]
assert min(5, 3) == 3
assert Solution().kSmallestPairs([1, 2], [3], 2) == [[1, 3], [2, 3]]

# Bài 2
nums1, nums2 = [1, 2], [3, 4]
matrix = [[u + v for v in nums2] for u in nums1]
assert matrix == [[4, 5], [5, 6]]
ans = Solution().kSmallestPairs(nums1, nums2, 3)
assert is_valid_answer(nums1, nums2, 3, ans)
assert sorted(map(tuple, ans)) == [(1, 3), (1, 4), (2, 3)]


# Bài 3
def k_smallest_pairs_todo_ans(nums1, nums2, k):
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
    heapq.heapify(heap)
    result = []
    while heap and len(result) < k:
        s, i, j = heapq.heappop(heap)
        result.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return result


for nums1, nums2, k, expected in examples:
    assert is_valid_answer(nums1, nums2, k, k_smallest_pairs_todo_ans(nums1, nums2, k))


# Bài 4
def k_smallest_pairs_visited2(nums1, nums2, k):
    heap, seen, out = [(nums1[0] + nums2[0], 0, 0)], {(0, 0)}, []
    while heap and len(out) < k:
        s, i, j = heapq.heappop(heap)
        out.append([nums1[i], nums2[j]])
        for a, b in ((i + 1, j), (i, j + 1)):
            if a < len(nums1) and b < len(nums2) and (a, b) not in seen:
                seen.add((a, b))
                heapq.heappush(heap, (nums1[a] + nums2[b], a, b))
    return out


for nums1, nums2, k, expected in examples:
    assert is_valid_answer(nums1, nums2, k, k_smallest_pairs_visited2(nums1, nums2, k))


# Bài 5(a)
def kth_smallest_in_sorted_matrix(matrix, k):
    n = len(matrix)
    heap = [(matrix[i][0], i, 0) for i in range(min(k, n))]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        if j + 1 < n:
            heapq.heappush(heap, (matrix[i][j + 1], i, j + 1))
    return heap[0][0]


mat = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
assert [kth_smallest_in_sorted_matrix(mat, k) for k in range(1, 10)] == sorted(x for row in mat for x in row)


# Bài 5(b)
def k_pairs_with_sum_at_most(nums1, nums2, T):
    count, j = 0, len(nums2) - 1
    for u in nums1:
        while j >= 0 and u + nums2[j] > T:
            j -= 1
        if j < 0:
            break
        count += j + 1
    return count


rng4 = random.Random(11)
for _ in range(300):
    a = sorted(rng4.randint(-8, 8) for _ in range(rng4.randint(1, 7)))
    b = sorted(rng4.randint(-8, 8) for _ in range(rng4.randint(1, 7)))
    T = rng4.randint(-16, 16)
    assert k_pairs_with_sum_at_most(a, b, T) == sum(1 for x in a for y in b if x + y <= T)
print("Đáp án bài 1-5 đều đúng")

Đáp án bài 1-5 đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`k` phần tử nhỏ nhất từ các dãy tăng: min-heap chứa phần tử đầu mỗi hàng)

```python
class Solution:
    def kSmallestPairs(self, nums1: List[int], nums2: List[int], k: int) -> List[List[int]]:
        m, n = len(nums1), len(nums2)
        heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, m))]
        heapq.heapify(heap)
        result = []
        while heap and len(result) < k:
            _, i, j = heapq.heappop(heap)
            result.append([nums1[i], nums2[j]])
            if j + 1 < n:
                heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
        return result
```

**Checklist**

- [ ] Heap chứa **tuple** `(tổng, i, j)` để so sánh theo tổng và biết cần mở rộng ở đâu?
- [ ] Khởi tạo **chỉ `min(k, m)` hàng**, không đẩy cả `m · n` cặp?
- [ ] Có kiểm tra `j + 1 < n` trước khi push?
- [ ] Vòng lặp dừng khi `len(result) == k` (và `heap` còn phần tử)?
- [ ] Nếu mở rộng hai hướng từ `(0, 0)`: đã dùng `visited`?
- [ ] Đã thử `k = 1`, `k = m·n`, mảng một phần tử, nhiều giá trị trùng, số âm?
- [ ] Test bằng **hàm kiểm tra hợp lệ** chứ không dùng `==` với thứ tự cặp cùng tổng?